In [ ]:
%pip install xgboost lightgbm catboost scikit-learn pandas pyarrow

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore') # Hides annoying warning messages

print("1. Loading Data & Feature Engineering...")
train_df = pd.read_csv('train.csv', engine='pyarrow')
test_df = pd.read_csv('test.csv', engine='pyarrow')

# Feature Engineering: Extract Hour and Minute from timestamp
for df in [train_df, test_df]:
    df['hour'] = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute'] = df['timestamp'].str.split(':').str[1].astype(int)

print("2. Preparing Data for All Algorithms...")
X = train_df.drop(columns=['Index', 'demand', 'timestamp'], errors='ignore')
y = train_df['demand'].astype('float32')
X_test = test_df.drop(columns=['Index', 'timestamp'], errors='ignore')

# Handle Missing Values
categorical_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
numerical_cols = ['day', 'NumberofLanes', 'Temperature', 'hour', 'minute']

# Fill empty categories with 'Missing' and empty numbers with the median
for col in categorical_cols:
    X[col] = X[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)
    
for col in numerical_cols:
    if col in X.columns:
        median_val = X[col].median()
        X[col] = X[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

# Encode Text to Numbers so XGBoost and Random Forest can read them
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[categorical_cols] = encoder.fit_transform(X[categorical_cols])
X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])

# Split into Training and Validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n3. 🏁 STARTING ALGORITHM SHOWDOWN 🏁")
print("Training 4 models... (This might take a minute)\n")

# Define the models with standard "good" default settings
models = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, verbose=-1),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=False, random_seed=42)
}

results = []

# Train and evaluate each model
for name, model in models.items():
    start_train = time.time()
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict and Score
    predictions = model.predict(X_val)
    r2 = r2_score(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    train_time = time.time() - start_train
    
    results.append({
        "Model": name,
        "Efficiency (R2)": r2 * 100,
        "RMSE": rmse,
        "Time (sec)": train_time
    })
    print(f"✅ {name} finished in {train_time:.1f} seconds.")

print("\n" + "="*55)
print("🏆 FINAL LEADERBOARD (Ranked by Efficiency)")
print("="*55)

# Sort results to find the winner
results_df = pd.DataFrame(results).sort_values(by="Efficiency (R2)", ascending=False).reset_index(drop=True)

for index, row in results_df.iterrows():
    medal = "🥇" if index == 0 else "🥈" if index == 1 else "🥉" if index == 2 else "  "
    print(f"{medal} {row['Model']:<15} | Efficiency: {row['Efficiency (R2)']:.2f}% | RMSE: {row['RMSE']:<6.4f} | Time: {row['Time (sec)']:.1f}s")
print("="*55 + "\n")

# Automatically generate submission for the winning model!
best_model_name = results_df.iloc[0]['Model']
print(f"Generating test predictions using the winner: {best_model_name}...")

best_model = models[best_model_name]
test_predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    'Index': test_df['Index'], 
    'demand': test_predictions
})
submission.to_csv(f'best_model_{best_model_name.lower().replace(" ", "_")}_submission.csv', index=False)
print(f"Done! Saved predictions to 'best_model_{best_model_name.lower().replace(' ', '_')}_submission.csv'")